# Point Cloud Classification — Main Workflow

This notebook is the cleaned version of the original `main_2` workflow.

The workflow:
1. Loads labeled point-cloud chunks.
2. Builds KNN contextual features.
3. Splits the data by complete clusters.
4. Uses spatially grouped cross-validation for hyperparameter tuning.
5. Trains one of the interchangeable classifiers defined in `models.py`.
6. Evaluates the model and exports it.
7. Optionally prepares classified point-cloud data for visualization.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from config import (
    CLASS_NAMES,
    CLASS_COLORS,
    MODEL_NAME,
    N_NEIGHBORS,
    N_RANDOM_SEARCH_ITERATIONS,
    TRAIN_FILES,
    TEST_FILES,
)

from data_utils import load_point_cloud_files
from features import build_features, build_tile_ids
from evaluation import (
    train_model,
    evaluate_model,
    print_confusion_matrix,
    save_metrics,
    save_model,
)
from visualization import add_classification_colors, build_prediction_dataframe


## 1. Load and prepare the labeled point clouds

In [ ]:
train_clusters = [item["cluster"] for item in TRAIN_FILES]
test_clusters = [item["cluster"] for item in TEST_FILES]

train_df = load_point_cloud_files(TRAIN_FILES)
test_df = load_point_cloud_files(TEST_FILES)

print(f"Training points: {len(train_df):,}")
print(f"Test points:     {len(test_df):,}")
print("\nTraining class distribution:")
print(train_df["Classification"].value_counts().sort_index())


## 2. Build KNN contextual features

In [ ]:
X_train = build_features(train_df, n_neighbors=N_NEIGHBORS)
X_test = build_features(test_df, n_neighbors=N_NEIGHBORS)

y_train = train_df["Classification"].to_numpy(dtype=int)
y_test = test_df["Classification"].to_numpy(dtype=int)

# Spatial tiles are used as groups during cross-validation.
groups = build_tile_ids(
    train_df[["X", "Y", "Z"]].to_numpy()
)

print("X_train shape:", X_train.shape)
print("X_test shape: ", X_test.shape)
print("Number of CV groups:", len(np.unique(groups)))


## 3. Select the classifier

Change `MODEL_NAME` in `config.py` to switch between:

- `xgboost`
- `random_forest`
- `extra_trees`

The training and evaluation code remains unchanged.


In [ ]:
print("Selected model:", MODEL_NAME)
print("Number of classes:", len(np.unique(y_train)))


## 4. Hyperparameter tuning and training

In [ ]:
search = train_model(
    model_name=MODEL_NAME,
    X_train=X_train,
    y_train=y_train,
    groups=groups,
    n_classes=len(np.unique(y_train)),
    random_search=True,
    n_iter=N_RANDOM_SEARCH_ITERATIONS,
)

print("\nBest parameters:")
print(search.best_params_)

print("\nBest cross-validation Macro F1:")
print(search.best_score_)

final_model = search.best_estimator_


In [ ]:
model_path = save_model(final_model, MODEL_NAME)

print("Saved model:", model_path)


## 5. Test-set evaluation

In [ ]:
y_test_pred, metrics = evaluate_model(
    final_model,
    X_test,
    y_test,
    class_names=CLASS_NAMES,
)

print_confusion_matrix(y_test, y_test_pred)

metrics_path = save_metrics(MODEL_NAME, metrics)

print("\nSaved metrics:", metrics_path)


## 6. Prepare the classified point cloud

In [ ]:
classified_test = build_prediction_dataframe(X_test, y_test_pred)
classified_test = add_classification_colors(
    classified_test,
    CLASS_COLORS,
)

classified_test.head()


In [ ]:
output_path = Path("outputs") / f"{MODEL_NAME}_classified_test.csv"
classified_test.to_csv(output_path, index=False)

print("Saved classified point cloud:", output_path)


## 7. Optional Open3D visualization

Run this section only when Open3D visualization is needed.


In [ ]:
import open3d as o3d

from visualization import point_cloud_arrays

xyz, rgb = point_cloud_arrays(classified_test)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(xyz)
pcd.colors = o3d.utility.Vector3dVector(rgb)

o3d.visualization.draw_geometries([pcd])


## 8. Generalization on an unlabeled point cloud

For a new unlabeled point cloud, repeat the feature-engineering step with
`build_features()` and call `final_model.predict(X_new)`.

The input must contain the same CloudCompare feature columns used by the
training data.
